# Lab 05 - N-gram Language Models

**Name:** Zain Ul Wahaj
**Roll Number:** FA23-BAI-058
**Course:** Natural Language Processing

---

This notebook contains the complete solution to all **9 tasks** of Lab 05.
Each part is solved in its own code cell, directly below the question it answers,
with its output shown underneath.

Everything uses the Python standard library only, natural logarithm (`ln`), and values rounded to 4 decimals.

### Contents

1. **Task 01** - Tokenization
2. **Task 02** - Generating N-grams
3. **Task 03** - Counting N-grams
4. **Task 04** - Unigram model
5. **Task 05** - Bigram model (MLE)
6. **Task 06** - Trigram model
7. **Task 07** - Zero-probability problem
8. **Task 08** - Smoothing
9. **Task 09** - Perplexity and model comparison

In [1]:
# Zain Ul Wahaj | FA23-BAI-058
# Natural Language Processing - Lab 05 - N-gram Language Models

NAME = "Zain Ul Wahaj"
ROLL_NUMBER = "FA23-BAI-058"

print("Name        :", NAME)
print("Roll Number :", ROLL_NUMBER)
print("Course      :", "Natural Language Processing")
print("Lab         :", "Lab 05 - N-gram Language Models")

import math
import itertools
from collections import Counter

# The data given in the lab sheet
train = ["the cat sat on the mat", "the cat ate the fish",
         "the dog sat on the rug", "the dog ate the bone"]
test = ["the cat sat on the rug", "the fish sat on the mat"]

print("\nTraining sentences:", len(train), "| Test sentences:", len(test))

Name        : Zain Ul Wahaj
Roll Number : FA23-BAI-058
Course      : Natural Language Processing
Lab         : Lab 05 - N-gram Language Models

Training sentences: 4 | Test sentences: 2


---

## Task 01: Tokenization

**Topic:** Splitting sentences into tokens and building the vocabulary

### Task 01 (1)

> Write tokenize(sentence) that lowercases, splits on spaces and adds <s> and </s>.

In [2]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 01 (1)
def tokenize(sentence):
    """lowercase -> split on spaces -> wrap with the start and end markers"""
    return ["<s>"] + sentence.lower().split() + ["</s>"]


print(tokenize("The cat sat on the mat"))

['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']


### Task 01 (2)

> Apply it to every training and test sentence and print the results.

In [3]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 01 (2)
train_tokens = [tokenize(s) for s in train]
test_tokens = [tokenize(s) for s in test]

print("TRAINING")
for toks in train_tokens:
    print("  ", toks)
print("TEST")
for toks in test_tokens:
    print("  ", toks)

TRAINING
   ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']
   ['<s>', 'the', 'cat', 'ate', 'the', 'fish', '</s>']
   ['<s>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '</s>']
   ['<s>', 'the', 'dog', 'ate', 'the', 'bone', '</s>']
TEST
   ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '</s>']
   ['<s>', 'the', 'fish', 'sat', 'on', 'the', 'mat', '</s>']


### Task 01 (3)

> Print the total number of tokens in the training corpus with markers and without markers.

In [4]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 01 (3)
with_markers = sum(len(t) for t in train_tokens)
without_markers = sum(len(t) - 2 for t in train_tokens)     # drop <s> and </s> of each sentence

print("Training tokens with markers    :", with_markers)
print("Training tokens without markers :", without_markers)
print("Difference                      :", with_markers - without_markers, "= 2 markers x", len(train), "sentences")

Training tokens with markers    : 30
Training tokens without markers : 22
Difference                      : 8 = 2 markers x 4 sentences


### Task 01 (4)

> Write build_vocab(sentences) and print V and the sorted vocabulary.

In [5]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 01 (4)
def build_vocab(sentences):
    """V = every word in the sentences plus </s>. <s> is never predicted, so it is not in V."""
    vocab = set()
    for s in sentences:
        vocab.update(s.lower().split())
    vocab.add("</s>")
    return vocab


vocab = build_vocab(train)
V = len(vocab)
print("V =", V)
print("Vocabulary:", sorted(vocab))

V = 11
Vocabulary: ['</s>', 'ate', 'bone', 'cat', 'dog', 'fish', 'mat', 'on', 'rug', 'sat', 'the']


---

## Task 02: Generating N-grams

**Topic:** Turning a token list into n-grams

### Task 02 (1)

> Write get_ngrams(tokens, n) that returns a list of tuples.

In [6]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 02 (1)
def get_ngrams(tokens, n):
    """All runs of n consecutive tokens, as tuples. Gives [] if n is larger than the sentence."""
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


print(get_ngrams(["a", "b", "c", "d"], 2))

[('a', 'b'), ('b', 'c'), ('c', 'd')]


### Task 02 (2)

> Print the unigrams, bigrams and trigrams of the first training sentence.

In [7]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 02 (2)
first = train_tokens[0]
print("Sentence:", first, "\n")
for n, name in [(1, "Unigrams"), (2, "Bigrams"), (3, "Trigrams")]:
    grams = get_ngrams(first, n)
    print(f"{name} ({len(grams)}):")
    print("  ", grams)

Sentence: ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>'] 

Unigrams (8):
   [('<s>',), ('the',), ('cat',), ('sat',), ('on',), ('the',), ('mat',), ('</s>',)]
Bigrams (7):
   [('<s>', 'the'), ('the', 'cat'), ('cat', 'sat'), ('sat', 'on'), ('on', 'the'), ('the', 'mat'), ('mat', '</s>')]
Trigrams (6):
   [('<s>', 'the', 'cat'), ('the', 'cat', 'sat'), ('cat', 'sat', 'on'), ('sat', 'on', 'the'), ('on', 'the', 'mat'), ('the', 'mat', '</s>')]


### Task 02 (3)

> For every sentence, verify with code that m tokens give m - n + 1 N-grams for n = 1, 2 and 3.

In [8]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 02 (3)
all_ok = True
for toks in train_tokens + test_tokens:
    m = len(toks)
    for n in (1, 2, 3):
        got = len(get_ngrams(toks, n))
        expected = m - n + 1
        if got != expected:
            all_ok = False
        print(f"m = {m}, n = {n}: got {got}, expected m - n + 1 = {expected}  {'OK' if got == expected else 'WRONG'}")
print("\nAll sentences passed:", all_ok)

m = 8, n = 1: got 8, expected m - n + 1 = 8  OK
m = 8, n = 2: got 7, expected m - n + 1 = 7  OK
m = 8, n = 3: got 6, expected m - n + 1 = 6  OK
m = 7, n = 1: got 7, expected m - n + 1 = 7  OK
m = 7, n = 2: got 6, expected m - n + 1 = 6  OK
m = 7, n = 3: got 5, expected m - n + 1 = 5  OK
m = 8, n = 1: got 8, expected m - n + 1 = 8  OK
m = 8, n = 2: got 7, expected m - n + 1 = 7  OK
m = 8, n = 3: got 6, expected m - n + 1 = 6  OK
m = 7, n = 1: got 7, expected m - n + 1 = 7  OK
m = 7, n = 2: got 6, expected m - n + 1 = 6  OK
m = 7, n = 3: got 5, expected m - n + 1 = 5  OK
m = 8, n = 1: got 8, expected m - n + 1 = 8  OK
m = 8, n = 2: got 7, expected m - n + 1 = 7  OK
m = 8, n = 3: got 6, expected m - n + 1 = 6  OK
m = 8, n = 1: got 8, expected m - n + 1 = 8  OK
m = 8, n = 2: got 7, expected m - n + 1 = 7  OK
m = 8, n = 3: got 6, expected m - n + 1 = 6  OK

All sentences passed: True


### Task 02 (4)

> Show what get_ngrams returns when n is larger than the sentence length, and make sure it does not crash.

In [9]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 02 (4)
short = ["<s>", "hello", "</s>"]                 # 3 tokens
for n in (3, 4, 10):
    print(f"n = {n:>2} on a 3-token sentence ->", get_ngrams(short, n))
print("n = 4 on an empty list           ->", get_ngrams([], 4))
print("\nNo crash: when n > m the range() is empty, so the result is an empty list.")

n =  3 on a 3-token sentence -> [('<s>', 'hello', '</s>')]
n =  4 on a 3-token sentence -> []
n = 10 on a 3-token sentence -> []
n = 4 on an empty list           -> []

No crash: when n > m the range() is empty, so the result is an empty list.


---

## Task 03: Counting N-grams

**Topic:** Counting n-grams with collections.Counter

### Task 03 (1)

> Use collections.Counter to build unigram, bigram and trigram counts for the training corpus.

In [10]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 03 (1)
unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()

for toks in train_tokens:
    unigram_counts.update(get_ngrams(toks, 1))
    bigram_counts.update(get_ngrams(toks, 2))
    trigram_counts.update(get_ngrams(toks, 3))

print("Unigram types:", len(unigram_counts), "| total:", sum(unigram_counts.values()))
print("Bigram types :", len(bigram_counts), "| total:", sum(bigram_counts.values()))
print("Trigram types:", len(trigram_counts), "| total:", sum(trigram_counts.values()))

Unigram types: 12 | total: 30
Bigram types : 18 | total: 26
Trigram types: 19 | total: 22


### Task 03 (2)

> Print each count table sorted from highest to lowest count.

In [11]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 03 (2)
def print_table(title, counter):
    print(f"{title} ({len(counter)} distinct)")
    # highest count first; ties alphabetical so the output is the same on every run
    for gram, c in sorted(counter.items(), key=lambda kv: (-kv[1], kv[0])):
        print(f"   {' '.join(gram):<22} {c}")
    print()


print_table("UNIGRAMS", unigram_counts)
print_table("BIGRAMS", bigram_counts)
print_table("TRIGRAMS", trigram_counts)

UNIGRAMS (12 distinct)
   the                    8
   </s>                   4
   <s>                    4
   ate                    2
   cat                    2
   dog                    2
   on                     2
   sat                    2
   bone                   1
   fish                   1
   mat                    1
   rug                    1

BIGRAMS (18 distinct)
   <s> the                4
   ate the                2
   on the                 2
   sat on                 2
   the cat                2
   the dog                2
   bone </s>              1
   cat ate                1
   cat sat                1
   dog ate                1
   dog sat                1
   fish </s>              1
   mat </s>               1
   rug </s>               1
   the bone               1
   the fish               1
   the mat                1
   the rug                1

TRIGRAMS (19 distinct)
   <s> the cat            2
   <s> the dog            2
   sat on the             2
   ate

### Task 03 (3)

> Print the 5 most common bigrams and the number of distinct unigrams, bigrams and trigrams.

In [12]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 03 (3)
top5 = sorted(bigram_counts.items(), key=lambda kv: (-kv[1], kv[0]))[:5]
print("5 most common bigrams:")
for gram, c in top5:
    print(f"   {gram}  ->  {c}")

print("\nDistinct unigrams:", len(unigram_counts))
print("Distinct bigrams :", len(bigram_counts))
print("Distinct trigrams:", len(trigram_counts))

5 most common bigrams:
   ('<s>', 'the')  ->  4
   ('ate', 'the')  ->  2
   ('on', 'the')  ->  2
   ('sat', 'on')  ->  2
   ('the', 'cat')  ->  2

Distinct unigrams: 12
Distinct bigrams : 18
Distinct trigrams: 19


### Task 03 (4)

> Write count_of(ngram) that returns 0 for an unseen N-gram instead of raising an error. Test it on one seen and one unseen bigram.

In [13]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 03 (4)
def count_of(ngram):
    """Count of a unigram/bigram/trigram tuple in the training data; 0 if never seen."""
    counter = {1: unigram_counts, 2: bigram_counts, 3: trigram_counts}[len(ngram)]
    return counter.get(ngram, 0)          # .get() never raises KeyError


print("seen   ('the', 'cat')  ->", count_of(("the", "cat")))
print("unseen ('cat', 'dog')  ->", count_of(("cat", "dog")))
print("unseen ('fish', 'sat') ->", count_of(("fish", "sat")))

seen   ('the', 'cat')  -> 2
unseen ('cat', 'dog')  -> 0
unseen ('fish', 'sat') -> 0


---

## Task 04: Unigram model

**Topic:** Unigram language model

### Task 04 (1)

> Write unigram_prob(w) using MLE, P(w) = C(w) / N.

In [14]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 04 (1)
# N counts the tokens the model can predict: every word and </s>, but not <s>
N = sum(c for (w,), c in unigram_counts.items() if w != "<s>")


def unigram_prob(w):
    """MLE unigram probability P(w) = C(w) / N."""
    return count_of((w,)) / N


print("N =", N, "(22 words + 4 end markers)")
print("P(the) =", round(unigram_prob("the"), 4), "  = 8 /", N)
print("P(cat) =", round(unigram_prob("cat"), 4), "  = 2 /", N)

N = 26 (22 words + 4 end markers)
P(the) = 0.3077   = 8 / 26
P(cat) = 0.0769   = 2 / 26


### Task 04 (2)

> Print all unigram probabilities as a table and assert that they sum to 1.

In [15]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 04 (2)
print(f"{'word':<8}{'count':>6}{'P(word)':>10}")
print("-" * 24)
for w in sorted(vocab, key=lambda w: (-count_of((w,)), w)):
    print(f"{w:<8}{count_of((w,)):>6}{unigram_prob(w):>10.4f}")

total = sum(unigram_prob(w) for w in vocab)
assert math.isclose(total, 1.0), total
print("\nSum of all probabilities =", round(total, 4), "-> assert passed")

word     count   P(word)
------------------------
the          8    0.3077
</s>         4    0.1538
ate          2    0.0769
cat          2    0.0769
dog          2    0.0769
on           2    0.0769
sat          2    0.0769
bone         1    0.0385
fish         1    0.0385
mat          1    0.0385
rug          1    0.0385

Sum of all probabilities = 1.0 -> assert passed


### Task 04 (3)

> Write unigram_sentence_prob(sentence) and print the result for both test sentences.

In [16]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 04 (3)
def unigram_sentence_prob(sentence):
    """Product of P(w) for every word and the final </s>; the words are treated as independent."""
    p = 1.0
    for w in tokenize(sentence)[1:]:          # skip <s>
        p *= unigram_prob(w)
    return p


for s in test:
    print(f'"{s}"')
    # the value is tiny, so scientific notation is used instead of rounding it to 0.0000
    print("   unigram probability =", f"{unigram_sentence_prob(s):.4e}")

"the cat sat on the rug"
   unigram probability = 2.5499e-07
"the fish sat on the mat"
   unigram probability = 1.2749e-07


### Task 04 (4)

> Use itertools.permutations on the words of the first test sentence and show that different word orders can give the same unigram probability.

In [17]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 04 (4)
words = test[0].split()                          # the cat sat on the rug
orders = set(itertools.permutations(words))      # set: "the" appears twice, so duplicates are removed

probs = {}
for order in orders:
    p = unigram_sentence_prob(" ".join(order))
    probs.setdefault(round(p, 12), []).append(" ".join(order))

print("Different word orders:", len(orders))
print("Different probabilities:", len(probs))
print("The original sentence:", test[0], "->", f"{unigram_sentence_prob(test[0]):.4e}")

some = list(orders)[:4]
print("\nFour other orders of the same words:")
for order in some:
    print(f"   {' '.join(order):<25} {unigram_sentence_prob(' '.join(order)):.4e}")
print("\nEvery order has the same probability, because multiplication does not care about order.")
print("A unigram model cannot tell 'the cat sat on the rug' from 'rug the on sat cat the'.")

Different word orders: 360
Different probabilities: 1
The original sentence: the cat sat on the rug -> 2.5499e-07

Four other orders of the same words:
   the cat the rug on sat    2.5499e-07
   the sat the cat on rug    2.5499e-07
   cat sat the the rug on    2.5499e-07
   cat the rug the sat on    2.5499e-07

Every order has the same probability, because multiplication does not care about order.
A unigram model cannot tell 'the cat sat on the rug' from 'rug the on sat cat the'.


---

## Task 05: Bigram model (MLE)

**Topic:** Bigram language model with maximum likelihood estimates

### Task 05 (1)

> Write bigram_prob(prev, w) using P(w | v) = C(v, w) / C(v).

In [18]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 05 (1)
def bigram_prob(prev, w):
    """MLE bigram probability P(w | prev) = C(prev, w) / C(prev). Returns 0.0 if prev was never seen."""
    c_prev = count_of((prev,))
    if c_prev == 0:
        return 0.0
    return count_of((prev, w)) / c_prev


print("P(cat | the)  =", round(bigram_prob("the", "cat"), 4), " = 2 / 8")
print("P(sat | cat)  =", round(bigram_prob("cat", "sat"), 4), " = 1 / 2")
print("P(the | <s>)  =", round(bigram_prob("<s>", "the"), 4), " = 4 / 4")

P(cat | the)  = 0.25  = 2 / 8
P(sat | cat)  = 0.5  = 1 / 2
P(the | <s>)  = 1.0  = 4 / 4


### Task 05 (2)

> Write next_word_distribution(prev) that returns a dictionary of next words and probabilities. Print it for "the" and assert that it sums to 1.

In [19]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 05 (2)
def next_word_distribution(prev):
    """{next word: P(next word | prev)} for every word that followed prev in training."""
    return {w2: bigram_prob(prev, w2) for (w1, w2) in bigram_counts if w1 == prev}


dist = next_word_distribution("the")
for w, p in sorted(dist.items(), key=lambda kv: (-kv[1], kv[0])):
    print(f"   P({w:<5}| the) = {p:.4f}")

total = sum(dist.values())
assert math.isclose(total, 1.0), total
print("\nSum =", round(total, 4), "-> assert passed")

   P(cat  | the) = 0.2500
   P(dog  | the) = 0.2500
   P(bone | the) = 0.1250
   P(fish | the) = 0.1250
   P(mat  | the) = 0.1250
   P(rug  | the) = 0.1250

Sum = 1.0 -> assert passed


### Task 05 (3)

> Write bigram_sentence_prob(sentence) using the chain rule and print the result for both test sentences.

In [20]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 05 (3)
def bigram_sentence_prob(sentence):
    """Chain rule with a one-word history: P(s) = product of P(w_i | w_(i-1))."""
    toks = tokenize(sentence)
    p = 1.0
    for prev, w in get_ngrams(toks, 2):
        p *= bigram_prob(prev, w)
    return p


for s in test:
    print(f'"{s}"')
    toks = tokenize(s)
    steps = [f"P({w}|{prev})={bigram_prob(prev, w):.4f}" for prev, w in get_ngrams(toks, 2)]
    print("   ", " x ".join(steps))
    print("    probability =", round(bigram_sentence_prob(s), 4), "\n")

"the cat sat on the rug"
    P(the|<s>)=1.0000 x P(cat|the)=0.2500 x P(sat|cat)=0.5000 x P(on|sat)=1.0000 x P(the|on)=1.0000 x P(rug|the)=0.1250 x P(</s>|rug)=1.0000
    probability = 0.0156 

"the fish sat on the mat"
    P(the|<s>)=1.0000 x P(fish|the)=0.1250 x P(sat|fish)=0.0000 x P(on|sat)=1.0000 x P(the|on)=1.0000 x P(mat|the)=0.1250 x P(</s>|mat)=1.0000
    probability = 0.0 



### Task 05 (4)

> Write predict_next(prev) that returns the most probable next word (break ties alphabetically) and test it on at least five different words.

In [21]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 05 (4)
def predict_next(prev):
    """Most probable next word after prev; the alphabetically first word wins a tie."""
    dist = next_word_distribution(prev)
    if not dist:
        return None                                    # prev never had a successor in training
    return min(dist, key=lambda w: (-dist[w], w))      # highest probability, then alphabetical


for prev in ["<s>", "the", "cat", "dog", "sat", "on", "ate", "fish", "mat", "elephant"]:
    dist = next_word_distribution(prev)
    guess = predict_next(prev)
    tied = bool(dist) and list(dist.values()).count(max(dist.values())) > 1
    print(f"after {prev:<9} -> {str(guess):<6}" + ("  (tie, alphabetical order used)" if tied else ""))

after <s>       -> the   
after the       -> cat     (tie, alphabetical order used)
after cat       -> ate     (tie, alphabetical order used)
after dog       -> ate     (tie, alphabetical order used)
after sat       -> on    
after on        -> the   
after ate       -> the   
after fish      -> </s>  
after mat       -> </s>  
after elephant  -> None  


---

## Task 06: Trigram model

**Topic:** Trigram language model

### Task 06 (1)

> Write trigram_prob(w1, w2, w3) using P(w3 | w1, w2) = C(w1, w2, w3) / C(w1, w2).

In [22]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 06 (1)
def trigram_prob(w1, w2, w3):
    """MLE trigram probability P(w3 | w1, w2). Returns 0.0 if the history (w1, w2) was never seen."""
    c_hist = count_of((w1, w2))
    if c_hist == 0:
        return 0.0
    return count_of((w1, w2, w3)) / c_hist


print("P(sat | the, cat)  =", round(trigram_prob("the", "cat", "sat"), 4), " = 1 / 2")
print("P(the | on, the)   =", round(trigram_prob("on", "the", "rug"), 4), " (P(rug | on, the) = 1 / 2)")
print("P(cat | <s>, the)  =", round(trigram_prob("<s>", "the", "cat"), 4), " = 2 / 4")

P(sat | the, cat)  = 0.5  = 1 / 2
P(the | on, the)   = 0.5  (P(rug | on, the) = 1 / 2)
P(cat | <s>, the)  = 0.5  = 2 / 4


### Task 06 (2)

> Write trigram_sentence_prob(sentence). Use the bigram model for the first word given <s> and the trigram model for all other words. Print the result for both test sentences.

In [23]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 06 (2)
def trigram_sentence_prob(sentence):
    """The first word has only <s> before it, so it uses the bigram P(w1 | <s>).
    Every later word uses the trigram P(w_i | w_(i-2), w_(i-1))."""
    toks = tokenize(sentence)
    p = bigram_prob("<s>", toks[1])
    for w1, w2, w3 in get_ngrams(toks, 3):
        p *= trigram_prob(w1, w2, w3)
    return p


for s in test:
    toks = tokenize(s)
    parts = [f"P({toks[1]}|<s>)={bigram_prob('<s>', toks[1]):.4f}"]
    parts += [f"P({c}|{a},{b})={trigram_prob(a, b, c):.4f}" for a, b, c in get_ngrams(toks, 3)]
    print(f'"{s}"')
    print("   ", " x ".join(parts))
    print("    probability =", round(trigram_sentence_prob(s), 4), "\n")

"the cat sat on the rug"
    P(the|<s>)=1.0000 x P(cat|<s>,the)=0.5000 x P(sat|the,cat)=0.5000 x P(on|cat,sat)=1.0000 x P(the|sat,on)=1.0000 x P(rug|on,the)=0.5000 x P(</s>|the,rug)=1.0000
    probability = 0.125 

"the fish sat on the mat"
    P(the|<s>)=1.0000 x P(fish|<s>,the)=0.0000 x P(sat|the,fish)=0.0000 x P(on|fish,sat)=0.0000 x P(the|sat,on)=1.0000 x P(mat|on,the)=0.5000 x P(</s>|the,mat)=1.0000
    probability = 0.0 



### Task 06 (3)

> Print the number of distinct unigrams, bigrams and trigrams in a small table.

In [24]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 06 (3)
print(f"{'n-gram':<10}{'distinct':>9}")
print("-" * 19)
for name, counter in [("unigram", unigram_counts), ("bigram", bigram_counts), ("trigram", trigram_counts)]:
    print(f"{name:<10}{len(counter):>9}")

n-gram     distinct
-------------------
unigram          12
bigram           18
trigram          19


### Task 06 (4)

> For each test sentence, list its unseen bigrams and its unseen trigrams, then print which model has more unseen N-grams.

In [25]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 06 (4)
for s in test:
    toks = tokenize(s)
    unseen_bi = [g for g in get_ngrams(toks, 2) if count_of(g) == 0]
    unseen_tri = [g for g in get_ngrams(toks, 3) if count_of(g) == 0]
    print(f'"{s}"')
    print(f"   unseen bigrams  ({len(unseen_bi)} of {len(get_ngrams(toks, 2))}):", unseen_bi)
    print(f"   unseen trigrams ({len(unseen_tri)} of {len(get_ngrams(toks, 3))}):", unseen_tri)
    if len(unseen_tri) > len(unseen_bi):
        print("   -> the trigram model has more unseen n-grams")
    elif len(unseen_tri) < len(unseen_bi):
        print("   -> the bigram model has more unseen n-grams")
    else:
        print("   -> both models have the same number of unseen n-grams")
    print()
print("Longer n-grams are more specific, so more of them were never seen in a tiny corpus.")

"the cat sat on the rug"
   unseen bigrams  (0 of 7): []
   unseen trigrams (0 of 6): []
   -> both models have the same number of unseen n-grams

"the fish sat on the mat"
   unseen bigrams  (1 of 7): [('fish', 'sat')]
   unseen trigrams (3 of 6): [('<s>', 'the', 'fish'), ('the', 'fish', 'sat'), ('fish', 'sat', 'on')]
   -> the trigram model has more unseen n-grams

Longer n-grams are more specific, so more of them were never seen in a tiny corpus.


---

## Task 07: Zero-probability problem

**Topic:** The zero-probability problem

### Task 07 (1)

> Write unseen_bigrams(sentence) that returns every bigram of the sentence that never occurs in training.

In [26]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 07 (1)
def unseen_bigrams(sentence):
    """Every bigram of the sentence (with markers) whose training count is 0."""
    return [g for g in get_ngrams(tokenize(sentence), 2) if count_of(g) == 0]


print(unseen_bigrams("the cat sat on the rug"))
print(unseen_bigrams("the fish sat on the mat"))

[]
[('fish', 'sat')]


### Task 07 (2)

> Run it on both test sentences and print the bigram MLE probability of each. Confirm that one of them is 0.

In [27]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 07 (2)
for s in test:
    print(f'"{s}"')
    print("   unseen bigrams  :", unseen_bigrams(s))
    print("   bigram MLE P(s) :", round(bigram_sentence_prob(s), 4))

zeros = [s for s in test if bigram_sentence_prob(s) == 0]
assert len(zeros) == 1
print(f'\nConfirmed: exactly one test sentence has probability 0 -> "{zeros[0]}"')
print("One unseen bigram makes the whole product 0, however likely the rest of the sentence is.")

"the cat sat on the rug"
   unseen bigrams  : []
   bigram MLE P(s) : 0.0156
"the fish sat on the mat"
   unseen bigrams  : [('fish', 'sat')]
   bigram MLE P(s) : 0.0

Confirmed: exactly one test sentence has probability 0 -> "the fish sat on the mat"
One unseen bigram makes the whole product 0, however likely the rest of the sentence is.


### Task 07 (3)

> Count how many of the possible word pairs (words only, markers excluded) never occur in training, and print this as a percentage.

In [28]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 07 (3)
words_only = sorted(w for w in vocab if w != "</s>")             # markers excluded
possible = [(a, b) for a in words_only for b in words_only]      # every ordered pair, including (w, w)
seen = [p for p in possible if count_of(p) > 0]
never = [p for p in possible if count_of(p) == 0]

print("Words (no markers)       :", len(words_only))
print("Possible word pairs      :", len(possible), f"= {len(words_only)} x {len(words_only)}")
print("Pairs seen in training   :", len(seen))
print("Pairs never seen         :", len(never))
print(f"Percentage never seen    : {len(never) / len(possible) * 100:.1f}%")

Words (no markers)       : 10
Possible word pairs      : 100 = 10 x 10
Pairs seen in training   : 13
Pairs never seen         : 87
Percentage never seen    : 87.0%


### Task 07 (4)

> Write safe_log(p) that returns negative infinity for p = 0 instead of raising a math error, and use it to compute the log probability of the second test sentence.

In [29]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 07 (4)
def safe_log(p):
    """Natural log that returns -inf for p = 0 (math.log(0) would raise a ValueError)."""
    return math.log(p) if p > 0 else float("-inf")


try:
    math.log(0)
except ValueError as e:
    print("math.log(0) raises ValueError:", e)

print("safe_log(0)   =", safe_log(0))
print("safe_log(0.5) =", round(safe_log(0.5), 4))

toks = tokenize(test[1])
log_p = sum(safe_log(bigram_prob(prev, w)) for prev, w in get_ngrams(toks, 2))
print(f'\nlog P("{test[1]}") =', log_p)

math.log(0) raises ValueError: math domain error
safe_log(0)   = -inf
safe_log(0.5) = -0.6931

log P("the fish sat on the mat") = -inf


---

## Task 08: Smoothing

**Topic:** Smoothing

### Task 08 (1)

> Write bigram_prob_addone(prev, w) using (C(v, w) + 1) / (C(v) + V).

In [30]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 08 (1)
def bigram_prob_addone(prev, w):
    """Add-one (Laplace) smoothing: pretend every possible bigram was seen once more."""
    return (count_of((prev, w)) + 1) / (count_of((prev,)) + V)


print("V =", V)
print("P_add1(sat | fish) =", round(bigram_prob_addone("fish", "sat"), 4), " = (0 + 1) / (1 + 11)")

V = 11
P_add1(sat | fish) = 0.0833  = (0 + 1) / (1 + 11)


### Task 08 (2)

> Print P(sat | fish) and P(cat | the) under both MLE and add-one smoothing, side by side.

In [31]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 08 (2)
print(f"{'probability':<18}{'MLE':>10}{'add-one':>10}")
print("-" * 38)
for prev, w in [("fish", "sat"), ("the", "cat")]:
    print(f"P({w} | {prev}){'':<{18 - len(f'P({w} | {prev})')}}{bigram_prob(prev, w):>10.4f}{bigram_prob_addone(prev, w):>10.4f}")

print("\nThe unseen bigram (fish sat) gets a small positive probability, while the common one")
print("(the cat) gives some of its probability away: 0.2500 -> 0.1579.")

probability              MLE   add-one
--------------------------------------
P(sat | fish)         0.0000    0.0833
P(cat | the)          0.2500    0.1579

The unseen bigram (fish sat) gets a small positive probability, while the common one
(the cat) gives some of its probability away: 0.2500 -> 0.1579.


### Task 08 (3)

> Loop over every possible previous word and assert that its smoothed next-word probabilities over all V words sum to 1.

In [32]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 08 (3)
# </s> is never a history (nothing follows the end of a sentence), so the possible previous words
# are <s> plus every word of the vocabulary except </s>
previous_words = ["<s>"] + sorted(w for w in vocab if w != "</s>")

for prev in previous_words:
    total = sum(bigram_prob_addone(prev, w) for w in sorted(vocab))      # over all V next words
    assert math.isclose(total, 1.0), (prev, total)
    print(f"previous word {prev:<6}: sum over {V} next words = {total:.4f}")
print("\nAll", len(previous_words), "previous words passed the assert.")

previous word <s>   : sum over 11 next words = 1.0000
previous word ate   : sum over 11 next words = 1.0000
previous word bone  : sum over 11 next words = 1.0000
previous word cat   : sum over 11 next words = 1.0000
previous word dog   : sum over 11 next words = 1.0000
previous word fish  : sum over 11 next words = 1.0000
previous word mat   : sum over 11 next words = 1.0000
previous word on    : sum over 11 next words = 1.0000
previous word rug   : sum over 11 next words = 1.0000
previous word sat   : sum over 11 next words = 1.0000
previous word the   : sum over 11 next words = 1.0000

All 11 previous words passed the assert.


### Task 08 (4)

> Write bigram_prob_addk(prev, w, k) and print P(sat | fish) for k = 1, 0.5, 0.1 and 0.01 in a table.

In [33]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 08 (4)
def bigram_prob_addk(prev, w, k):
    """Add-k smoothing: (C(v, w) + k) / (C(v) + k * V). k = 1 is add-one."""
    return (count_of((prev, w)) + k) / (count_of((prev,)) + k * V)


print(f"{'k':<8}{'P(sat | fish)':>15}")
print("-" * 23)
for k in (1, 0.5, 0.1, 0.01):
    print(f"{k:<8}{bigram_prob_addk('fish', 'sat', k):>15.4f}")
print("\nA smaller k moves less probability to unseen bigrams, so P(sat | fish) shrinks towards 0.")

k         P(sat | fish)
-----------------------
1                0.0833
0.5              0.0769
0.1              0.0476
0.01             0.0090

A smaller k moves less probability to unseen bigrams, so P(sat | fish) shrinks towards 0.


---

## Task 09: Perplexity and model comparison

**Topic:** Perplexity and comparing the models

### Task 09 (1)

> Write perplexity(sentences, prob_fn) that returns infinity when any probability is 0.

In [34]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 09 (1)
def perplexity(sentences, prob_fn):
    """Perplexity = exp(-(1/T) * sum of ln P) over all T predicted tokens in all the sentences.
    prob_fn(prev, w) gives P(w | prev). Every word and the final </s> is predicted; <s> is not.
    If any probability is 0 the perplexity is infinity."""
    total_log = 0.0
    T = 0
    for s in sentences:
        for prev, w in get_ngrams(tokenize(s), 2):
            p = prob_fn(prev, w)
            if p == 0:
                return float("inf")
            total_log += math.log(p)
            T += 1
    return math.exp(-total_log / T)


print("Tokens predicted in the two test sentences:", sum(len(tokenize(s)) - 1 for s in test), "(6 words + </s> each)")
print("Bigram MLE perplexity:", perplexity(test, bigram_prob))

Tokens predicted in the two test sentences: 14 (6 words + </s> each)
Bigram MLE perplexity: inf


### Task 09 (2)

> Compute the perplexity of the two test sentences together for the bigram MLE model, the bigram add-one model and the unigram add-one model, where P(w) = (C(w) + 1) / (N + V). Print one results table.

In [35]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 09 (2)
def unigram_prob_addone(prev, w):
    """Unigram add-one: P(w) = (C(w) + 1) / (N + V). The previous word is ignored."""
    return (count_of((w,)) + 1) / (N + V)


results = [
    ("bigram MLE", perplexity(test, bigram_prob)),
    ("bigram add-one", perplexity(test, bigram_prob_addone)),
    ("unigram add-one", perplexity(test, unigram_prob_addone)),
]

print(f"{'model':<18}{'perplexity':>12}")
print("-" * 30)
for name, pp in results:
    print(f"{name:<18}{pp:>12.4f}")

finite = [(n, pp) for n, pp in results if pp != float("inf")]
print("\nLowest finite perplexity:", min(finite, key=lambda r: r[1])[0])

model               perplexity
------------------------------
bigram MLE                 inf
bigram add-one          5.7969
unigram add-one         9.1370

Lowest finite perplexity: bigram add-one


### Task 09 (3)

> Compute the perplexity of the add-k bigram model for k = 1, 0.5, 0.1 and 0.01. Print the table and the best k.

In [36]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 09 (3)
ks = [1, 0.5, 0.1, 0.01]
k_results = {k: perplexity(test, lambda prev, w, k=k: bigram_prob_addk(prev, w, k)) for k in ks}

print(f"{'k':<8}{'perplexity':>12}")
print("-" * 20)
for k, pp in k_results.items():
    print(f"{k:<8}{pp:>12.4f}")

best_k_test = min(k_results, key=k_results.get)
print("\nBest k on the test sentences:", best_k_test, "-> perplexity", round(k_results[best_k_test], 4))

k         perplexity
--------------------
1             5.7969
0.5           4.5703
0.1           2.9559
0.01          2.6272

Best k on the test sentences: 0.01 -> perplexity 2.6272


### Task 09 (4)

> Train only on the first three training sentences (build V from all four) and pick the best k using the fourth sentence as validation data. Report the test perplexity with that k, and compare it with the best k found directly on the test set.

In [37]:
# Zain Ul Wahaj | FA23-BAI-058 | Task 09 (4)
# Counts from the first three training sentences only. V stays 11 (built from all four sentences).
train3 = train[:3]
validation = [train[3]]                                  # "the dog ate the bone"

uni3, bi3 = Counter(), Counter()
for s in train3:
    toks = tokenize(s)
    uni3.update(get_ngrams(toks, 1))
    bi3.update(get_ngrams(toks, 2))


def prob_addk_3(prev, w, k):
    return (bi3.get((prev, w), 0) + k) / (uni3.get((prev,), 0) + k * V)


val_pp = {k: perplexity(validation, lambda p, w, k=k: prob_addk_3(p, w, k)) for k in ks}
test_pp = {k: perplexity(test, lambda p, w, k=k: prob_addk_3(p, w, k)) for k in ks}

print("Trained on 3 sentences, V =", V)
print(f"{'k':<8}{'validation PP':>15}{'test PP':>12}")
print("-" * 35)
for k in ks:
    print(f"{k:<8}{val_pp[k]:>15.4f}{test_pp[k]:>12.4f}")

best_k_val = min(val_pp, key=val_pp.get)
best_k_direct = min(test_pp, key=test_pp.get)

print(f"\nBest k chosen on the validation sentence : {best_k_val}")
print(f"Test perplexity with that k              : {test_pp[best_k_val]:.4f}")
print(f"Best k found directly on the test set    : {best_k_direct}")
print(f"Test perplexity with that k              : {test_pp[best_k_direct]:.4f}")
if best_k_val == best_k_direct:
    print("\nBoth methods pick the same k, so tuning on the validation sentence cost nothing here.")
else:
    print("\nThe two k values differ. Picking k on the test set looks slightly better only because it")
    print("peeks at the test data; the validation choice is the honest estimate of unseen-data performance.")
print("(With a single validation sentence this choice is noisy - a bigger validation set would be more reliable.)")

Trained on 3 sentences, V = 11
k         validation PP     test PP
-----------------------------------
1                8.5857      5.7406
0.5              8.0163      4.4539
0.1              8.0260      2.7778
0.01            13.1304      2.4258

Best k chosen on the validation sentence : 0.5
Test perplexity with that k              : 4.4539
Best k found directly on the test set    : 0.01
Test perplexity with that k              : 2.4258

The two k values differ. Picking k on the test set looks slightly better only because it
peeks at the test data; the validation choice is the honest estimate of unseen-data performance.
(With a single validation sentence this choice is noisy - a bigger validation set would be more reliable.)


---

## End of Lab 05

All 9 tasks (36 parts) completed by **Zain Ul Wahaj** - **FA23-BAI-058**.